# Quick Start: 첫 시뮬레이션

**VFG Path Following 패키지 시작 가이드**

이 노트북에서는 `vfg_pathfollowing` 패키지를 사용하여 경로 추종 시뮬레이션을 실행하는 기본적인 방법을 소개합니다.

This notebook demonstrates the basic 5-line pattern for running a closed-loop path-following simulation using the Vector Field Guidance (VFG) framework with an LPV H-infinity controller.

**Author:** Suwon Lee, Kookmin University

In [ ]:
# -*- coding: utf-8 -*-
%matplotlib inline

## 1. 경로 생성 및 시뮬레이션 실행

`StepCurvaturePath`는 직선-원호-직선으로 구성된 경로입니다. 곡률이 갑자기 변하는 구간에서 제어기의 과도 응답을 평가하기에 적합합니다.

아래 코드 4줄로 경로 생성부터 시뮬레이션 실행까지 완료됩니다.

In [ ]:
# -*- coding: utf-8 -*-
from vfg_pathfollowing import Simulator, StepCurvaturePath, compute_metrics

# 경로 생성: 반지름 0.5m, 원호 각도 90도
path = StepCurvaturePath(R=0.5, theta_arc=1.57)

# 시뮬레이터 생성: LPV H-inf 제어기, 속도 1.5 m/s
sim = Simulator(path, controller='lpv-hinf', speed=1.5)

# 시뮬레이션 실행 (20초)
result = sim.run(T=20.0)

# 결과 시각화 (경로 + 오차 3-panel plot)
result.plot(path=path)

## 2. 결과 데이터 확인

`SimResult` 객체에는 시뮬레이션 전체 시계열 데이터가 저장되어 있습니다:
- `time`: 시간 벡터 [s]
- `e_psi`: 방위각 오차 [rad]
- `e_d`: 횡방향 오차 [m]
- `X`, `Y`: 차량 위치 궤적 [m]
- `delta`: 실제 조향각 [rad]

In [ ]:
import numpy as np

print(f"시뮬레이션 시간: {result.time[-1]:.1f} s")
print(f"데이터 포인트 수: {len(result.time)}")
print(f"방위각 오차 (e_psi) 범위: [{np.degrees(result.e_psi.min()):.2f}, {np.degrees(result.e_psi.max()):.2f}] deg")
print(f"횡방향 오차 (e_d)   범위: [{result.e_d.min():.4f}, {result.e_d.max():.4f}] m")
print(f"최대 조향각: {np.degrees(np.abs(result.delta).max()):.2f} deg")

## 3. 성능 지표 계산

`compute_metrics()`는 과도 구간(기본 2초)을 제외한 정상 상태 RMS 오차, 최대 오차, 정착 시간을 계산합니다.

In [ ]:
# 성능 지표 계산 (과도 구간 2초 제외)
metrics = compute_metrics(result, t_transient=2.0)

print("=== 성능 지표 ===")
print(f"RMS 방위각 오차 (정상상태): {metrics['rms_e_psi_deg']:.3f} deg")
print(f"RMS 횡방향 오차 (정상상태): {metrics['rms_e_d']:.4f} m")
print(f"최대 방위각 오차:           {metrics['max_e_psi_deg']:.3f} deg")
print(f"최대 횡방향 오차:           {metrics['max_e_d']:.4f} m")
print(f"정착 시간 (|e_psi| < 2 deg): {metrics['settling_time']:.2f} s")